In [1]:
# ==============================================================================
# Cell 1: Environment Setup & Excel Dataset Loading
# ==============================================================================

import numpy as np
import pandas as pd
import re
import math
from collections import defaultdict, Counter

# Set random seed for reproducibility
np.random.seed(42)

DATASET_FILE = "CLEANED_MULTILINGUAL_DATASET.xlsx"

try:
    df_all = pd.read_excel(DATASET_FILE, sheet_name='All_3500')
    print(f"Successfully loaded '{DATASET_FILE}' with {len(df_all)} records.")
except FileNotFoundError:
    print(f"File '{DATASET_FILE}' not found in runtime. Please upload the file.")
    from google.colab import files
    uploaded = files.upload()
    df_all = pd.read_excel(DATASET_FILE, sheet_name='All_3500')

# Extract predefined dataset splits
train_df = df_all[df_all['Split'] == 'train'].reset_index(drop=True)
val_df = df_all[df_all['Split'] == 'validation'].reset_index(drop=True)
test_df = df_all[df_all['Split'] == 'test'].reset_index(drop=True)

print(f"Dataset Splits -> Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

# Map target categories (7 unique categories) (Label Encoding)
categories = sorted(df_all['Category'].unique().tolist())
cat2id = {c: i for i, c in enumerate(categories)}
id2cat = {i: c for i, c in enumerate(categories)}

y_train = np.array([cat2id[c] for c in train_df['Category']])
y_val = np.array([cat2id[c] for c in val_df['Category']])
y_test = np.array([cat2id[c] for c in test_df['Category']])

print("\nCategories:", categories)

File 'CLEANED_MULTILINGUAL_DATASET.xlsx' not found in runtime. Please upload the file.


Saving CLEANED_MULTILINGUAL_DATASET.xlsx to CLEANED_MULTILINGUAL_DATASET.xlsx
Dataset Splits -> Train: 2779, Validation: 371, Test: 350

Categories: ['English', 'English + Hindi', 'English + Hindi + Kannada', 'English + Kannada', 'Hindi', 'Kannada', 'Kannada + Hindi']


In [2]:
# ==============================================================================
# Cell 2: Hardcoded Character N-Gram TF-IDF Vectorizer
# ==============================================================================

class HardcodedTfidfVectorizer:
    def __init__(self, ngram_range=(2, 3), max_features=1000):
        self.ngram_range = ngram_range
        self.max_features = max_features
        self.vocabulary_ = {}
        self.idf_ = None
# n grams (bigram and trigram)
    def _extract_ngrams(self, text):
        text = str(text).lower()
        ngrams = []
        for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
            for i in range(len(text) - n + 1):
                ngrams.append(text[i:i+n])
        return ngrams

    def fit_transform(self, raw_documents):
        doc_ngrams = [self._extract_ngrams(doc) for doc in raw_documents]

        freq_counter = Counter()
        for ngrams in doc_ngrams:
            freq_counter.update(set(ngrams))

        most_common = freq_counter.most_common(self.max_features)
        self.vocabulary_ = {ngram: i for i, (ngram, _) in enumerate(most_common)}

        N = len(raw_documents)
        df_count = np.zeros(len(self.vocabulary_))
        for ngrams in doc_ngrams:
            unique_in_doc = set(ngrams)
            for ngram in unique_in_doc:
                if ngram in self.vocabulary_:
                    df_count[self.vocabulary_[ngram]] += 1

        # Smoothed IDF formula: log((1 + N) / (1 + df)) + 1
        self.idf_ = np.log((1 + N) / (1 + df_count)) + 1.0
        return self.transform(raw_documents)

    def transform(self, raw_documents):
        num_docs = len(raw_documents)
        num_features = len(self.vocabulary_)
        X = np.zeros((num_docs, num_features))

        for d_idx, doc in enumerate(raw_documents):
            ngrams = self._extract_ngrams(doc)
            counts = Counter(ngrams)
            total = len(ngrams) if len(ngrams) > 0 else 1

            for ngram, count in counts.items():
                if ngram in self.vocabulary_:
                    f_idx = self.vocabulary_[ngram]
                    tf = count / total
                    X[d_idx, f_idx] = tf * self.idf_[f_idx]

            # L2 Row Normalization
            norm = np.linalg.norm(X[d_idx])
            if norm > 0:
                X[d_idx] = X[d_idx] / norm

        return X

# Feature Extraction
vectorizer = HardcodedTfidfVectorizer(ngram_range=(2, 3), max_features=1000)
X_train = vectorizer.fit_transform(train_df['Sentences'].tolist())
X_val = vectorizer.transform(val_df['Sentences'].tolist())
X_test = vectorizer.transform(test_df['Sentences'].tolist())

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (2779, 1000)
X_test shape: (350, 1000)


In [ ]:
# ==============================================================================
# Cell 3: Hardcoded ML Classifiers Built Ground-Up using NumPy
# ==============================================================================

class HardcodedMultinomialNB:
    def __init__(self, alpha=0.5):
        self.alpha = alpha

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        self.class_log_prior = np.zeros(n_classes)
        self.feature_log_prob = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.class_log_prior[idx] = np.log(X_c.shape[0] / float(n_samples))
            count_c = X_c.sum(axis=0) + self.alpha
            self.feature_log_prob[idx] = np.log(count_c / count_c.sum())

    def predict(self, X):
        jll = X @ self.feature_log_prob.T + self.class_log_prior
        return self.classes[np.argmax(jll, axis=1)]


class HardcodedLogisticRegression:
    def __init__(self, lr=0.5, n_iters=800):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.biases = None

    def _softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        y_onehot = np.zeros((n_samples, n_classes))
        y_onehot[np.arange(n_samples), y] = 1

        self.weights = np.zeros((n_features, n_classes))
        self.biases = np.zeros((1, n_classes))

        # Gradient Descent Loop
        for _ in range(self.n_iters):
            scores = X @ self.weights + self.biases
            probs = self._softmax(scores)

            dw = (1 / n_samples) * (X.T @ (probs - y_onehot))
            db = (1 / n_samples) * np.sum(probs - y_onehot, axis=0, keepdims=True)

            self.weights -= self.lr * dw
            self.biases -= self.lr * db

    def predict(self, X):
        scores = X @ self.weights + self.biases
        probs = self._softmax(scores)
        return self.classes[np.argmax(probs, axis=1)]


# Model Training
nb_model = HardcodedMultinomialNB(alpha=0.5)
nb_model.fit(X_train, y_train)

lr_model = HardcodedLogisticRegression(lr=0.5, n_iters=800)
lr_model.fit(X_train, y_train)

print("Training finished successfully!")

In [ ]:
# ==============================================================================
# Cell 4: Custom Evaluation Metrics & Graphical Confusion Matrix
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model(y_true, y_pred, model_name="Model"):
    acc = np.mean(y_true == y_pred)
    f1s = []

    print(f"\n==================== {model_name} Evaluation ====================")
    print(f"Overall Accuracy: {acc * 100:.2f}%\n")

    classes_present = sorted(list(set(y_true)))
    for c in classes_present:
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        f1s.append(f1)
        print(f"Class '{id2cat[c]}': Precision={prec:.2f}, Recall={rec:.2f}, F1={f1:.2f}")

    macro_f1 = np.mean(f1s)
    print(f"\nMacro-F1 Score: {macro_f1:.4f}")

    # Generate Confusion Matrix
    n_c = len(categories)
    cm = np.zeros((n_c, n_c), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    # Plot Graphical Confusion Matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=categories, yticklabels=categories)
    plt.title(f'Confusion Matrix: {model_name}')
    plt.ylabel('Actual Category')
    plt.xlabel('Predicted Category')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Evaluate baseline models on Test Set
evaluate_model(y_test, nb_model.predict(X_test), "Hardcoded Naive Bayes")
evaluate_model(y_test, lr_model.predict(X_test), "Hardcoded Logistic Regression")

In [ ]:
# ==============================================================================
# Cell 5: Word-Level Lexicon Extraction & Token Tagging
# ==============================================================================

word_lang_counts = defaultdict(Counter) #creating Lexicon

# Extract word token language associations directly from dataset ground truth
for token_str in train_df['Token_Lang_Tags'].dropna():
    tokens = str(token_str).split()
    for token in tokens:
        if '/' in token:
            parts = token.rsplit('/', 1)
            word = re.sub(r'[^\w\s]', '', parts[0].lower().strip())
            tag = parts[1].upper().strip()
            if word and tag in {'E', 'H', 'K'}:
                word_lang_counts[word][tag] += 1

tag2lang = {'E': 'English', 'H': 'Hindi', 'K': 'Kannada'}

def tag_word(word):
    clean_w = re.sub(r'[^\w\s]', '', word.lower().strip())
    if not clean_w:
        return "Punctuation/Symbol"

    # Vocabulary lookup
    if clean_w in word_lang_counts:
        top_tag = word_lang_counts[clean_w].most_common(1)[0][0]
        return tag2lang[top_tag]

    # Fallback to character n-gram prediction for unseen words
    w_vec = vectorizer.transform([clean_w])
    pred_cat = id2cat[lr_model.predict(w_vec)[0]]

    if "English" in pred_cat: return "English"
    if "Hindi" in pred_cat: return "Hindi"
    if "Kannada" in pred_cat: return "Kannada"
    return "English"

print(f"Extracted word lexicon size: {len(word_lang_counts)} unique tokens.")

In [ ]:
# ==============================================================================
# Cell 6: Sentence Analysis Engine (with Corrected Categorization logic)
# ==============================================================================

def analyze_sentence(sentence):
    print("=" * 75)
    print(f"Input Sentence: \"{sentence}\"")

    # 1. Word-by-Word Language Tagging(List comprehension)
    words = sentence.strip().split()
    word_tags = [tag_word(w) for w in words]

    # 2. Corrected Category Assignment based on actual word-level tags detected
    unique_langs = sorted(list(set(tag for tag in word_tags if tag not in {"Punctuation/Symbol"})))

    # Construct correct category label dynamically
    if len(unique_langs) == 1:
        predicted_category = unique_langs[0]
    elif len(unique_langs) == 2:
        if "English" in unique_langs and "Kannada" in unique_langs:
            predicted_category = "English + Kannada"
        elif "English" in unique_langs and "Hindi" in unique_langs:
            predicted_category = "English + Hindi"
        elif "Kannada" in unique_langs and "Hindi" in unique_langs:
            predicted_category = "Kannada + Hindi"
        else:
            predicted_category = " + ".join(unique_langs)
    elif len(unique_langs) >= 3:
        predicted_category = "English + Hindi + Kannada"
    else:
        # Fallback to model classifier
        vec = vectorizer.transform([sentence])
        pred_id = lr_model.predict(vec)[0]
        predicted_category = id2cat[pred_id]

    print(f"\nPredicted Category: [{predicted_category}]")

    print("\nWord-by-Word Language Tagging:")
    for w, t in zip(words, word_tags):
        print(f"  - '{w}': {t}")

    # 3. Detect Language Switch Points
    switch_points = []
    for i in range(1, len(word_tags)):
        prev_tag = word_tags[i-1]
        curr_tag = word_tags[i]
        if prev_tag != curr_tag and prev_tag != "Punctuation/Symbol" and curr_tag != "Punctuation/Symbol":
            switch_points.append((i, words[i], prev_tag, curr_tag))

    print("\nLanguage Switch Points:")
    if switch_points:
        for idx, word, from_lang, to_lang in switch_points:
            print(f"  * Language change at Word Index {idx} ('{word}'): Switched from [{from_lang}] -> [{to_lang}]")
    else:
        print("  * No language switch points detected. Monolingual sentence.")

    # 4. Trilingual Presence Check
    trilingual = {"English", "Hindi", "Kannada"}.issubset(set(unique_langs))

    print("Analysis:")
    if trilingual:
        print("  >>> SUCCESS: Sentence contains ALL THREE languages (English + Hindi + Kannada)! <<<")
    else:
        print(f"  * Languages present in sentence: {', '.join(unique_langs)}")
    print("=" * 75 + "\n")

In [ ]:
# ==============================================================================
# Cell 7: Test Inferences on Examples
# ==============================================================================


analyze_sentence("boy huduga ladka")